In [2]:
!pip install --upgrade pip setuptools wheel


In [3]:
!pip install pyarrow --only-binary=:all:


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 78.3 MB/s  0:00:00 eta 0:00:01


In [5]:
!pip install torch transformers accelerate sentencepiece
!pip install datasets pyarrow==14.0.2 --only-binary=:all:


  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.4.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.2.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

In [6]:
import pyarrow
import datasets

print(pyarrow.__version__)
print(datasets.__version__)


<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.NativeFile size changed, may indicate binary incompatibility. Expected 96 from C header, got 104 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.BufferedInputStream size changed, may indicate binary incompatibility. Expected 

20.0.0
2.19.2


In [3]:
# ============================================================
# FINAL HYDRALORA SCRIPT — NAN-SAFE, STABLE, SINGLE BLOCK
# Mistral-7B | A100 80GB | BF16 | Research-Grade
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

# =========================
# Tokenizer
# =========================
model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# Model
# =========================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# =========================
# HydraLoRA
# =========================
class HydraLoRA(nn.Module):
    def __init__(self, in_features, out_features, r=8, num_experts=8, alpha=32, dropout=0.05):
        super().__init__()
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        # ---- Stable router ----
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)
        gates = F.softmax(logits, dim=-1)

        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(B(self.dropout(A(x))) * self.scaling)

        expert_outs = torch.stack(expert_outs, dim=-2)
        return torch.sum(gates.unsqueeze(-1) * expert_outs, dim=-2)

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer

        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        )

        # Device + dtype alignment
        self.hydra.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# =========================
# Inject HydraLoRA into Q,V
# =========================
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# =========================
# Freeze non-Hydra params
# =========================
for name, param in model.named_parameters():
    param.requires_grad = ("hydra" in name.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# =========================
# Dataset
# =========================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# =========================
# Optimizer (stable LR)
# =========================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# =========================
# Training (NaN-SAFE)
# =========================
model.train()

GRAD_ACC = 8
MAX_STEPS = 500
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    # ---- Skip empty / tiny batches ----
    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    # ---- Skip NaN / Inf losses ----
    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"✅ Training finished | Skipped batches: {skipped}")


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.40s/it]


Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.5946
Step 50 | Loss: 2.6866
Step 100 | Loss: 2.0988
Step 150 | Loss: 2.2924
Step 200 | Loss: 1.6744
Step 250 | Loss: 2.8313
Step 300 | Loss: 1.6380
Step 350 | Loss: 1.2959
Step 400 | Loss: 1.8854
Step 450 | Loss: 5.6311
✅ Training finished | Skipped batches: 252


In [4]:
# Router entropy (lower over time = specialization)
with torch.no_grad():
    x = torch.randn(2, 16, model.config.hidden_size, device="cuda", dtype=torch.bfloat16)
    logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
    probs = torch.softmax(logits, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
    print("Router entropy:", entropy.item())


Router entropy: 2.078125


PAPER-FAITHFUL HYDRALORA CODE